In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated/checkpoints/post_cell_5.pickle

trying: ['responses_df_2022_cell10', 'responses_df_2022']


me:  10
me:  10
trying: ['responses_df_2022_cell10', 'responses_df_2022']


me:  10
me:  10
trying: ['file_name_2017', 'file_name_2018']
me:  2
me:  2
trying: ['file_name_2017', 'file_name_2018']
me:  2
me:  2
trying: ['base_dir_2019']
me:  2
trying: ['file_name_2022']
me:  2
trying: ['percentages_cell17']
me:  10
trying: ['responses_df_2020']


me:  2
trying: ['file_name_2021']
me:  2
trying: ['question_of_interest']
me:  10
trying: ['load_survey_data']
me:  2
trying: ['base_dir_2020']
me:  2
trying: ['file_name_2020']
me:  2
trying: ['directory_cell8']
me:  2
trying: ['percentages']
me:  10
trying: ['base_dir_2021']
me:  2
trying: ['file_name_2019']
me:  2
trying: ['px']
me:  0
trying: ['responses_df_2021']


me:  2
trying: ['directory']
me:  2
trying: ['base_dir_2017']
me:  2
trying: ['orientation_for_chart']
me:  10
trying: ['responses_df_2017']


...

me:  2
trying: ['orig_output']
me:  9
trying: ['responses_df_2018']


me:  2
trying: ['responses_df_2019_cell10']


me:  4
trying: ['replace_hyphen_with_en_dash']
me:  4
trying: ['np']
me:  0
trying: ['warnings']
me:  0
trying: ['go']
me:  0
trying: ['percentages_per_country_df']
me:  8
trying: ['create_dataframe_of_counts_16']
me:  8
trying: ['pd']
me:  0
trying: ['glob']
me:  0
trying: ['sns']
me:  0
trying: ['count_then_return_percent_17']
me:  10
trying: ['responses_df_2018_cell10']


me:  4
trying: ['base_dir_2022']
me:  2
trying: ['responses_in_order']
me:  10


Declaring variable px
Declaring variable np
Declaring variable warnings
Declaring variable go
Declaring variable pd
Declaring variable glob
Declaring variable sns
Declaring variable file_name_2017
Declaring variable file_name_2018
Declaring variable file_name_2017
Declaring variable file_name_2018
Declaring variable base_dir_2019
Declaring variable file_name_2022
Declaring variable responses_df_2020
Declaring variable file_name_2021
Declaring variable load_survey_data
Declaring variable base_dir_2020
Declaring variable file_name_2020
Declaring variable directory_cell8
Declaring variable base_dir_2021
Declaring variable file_name_2019
Declaring variable responses_df_2021
Declaring variable directory
Declaring variable base_dir_2017
Declaring variable responses_df_2017
Declaring variable base_dir_2018
Declaring variable responses_df_2019
Declaring variable responses_df_2018
Declaring variable base_dir_2022
Declaring variable responses_df_2019_cell10
Declaring variable replace_hyphen_with

In [4]:
%%RecordEvent
%%time
### cell 6 ###

### cell 6 optimized ###

# 0) Define the subset of countries used for bucketing
subset_of_countries = [
    'Other', 'India', 'United States of America', 'Brazil', 'Nigeria',
    'Pakistan', 'Japan', 'China', 'Egypt', 'Indonesia', 'Mexico',
    'Turkey', 'Russia'
]

# 1) Define the question and x-axis title
question_of_interest_cell18 = 'In which country do you currently reside?'
title_for_x_axis = ''

# 2) Optimize the count → percentage function
#    (preserve original denominator semantics: non-null count)
def count_then_return_percent_18(dataframe, x_axis_title):
    return (
        dataframe[x_axis_title]
        .value_counts(dropna=False)
        .mul(100)
        .div(dataframe[x_axis_title].count())
        .round(1)
    )

# 3a) Fix 2022 UK naming in place (values only)
responses_df_2022_cell10.replace(
    'United Kingdom (UK)',
    'United Kingdom of Great Britain and Northern Ireland',
    inplace=True
)

# 3b) Rename 2017 country column and remap values
responses_df_2017.rename(
    columns={
        'Country': question_of_interest_cell18,
        'CurrentJobTitleSelect':
            'Select the title most similar to your current role (or most recent title if retired): - Selected Choice'
    },
    inplace=True
)
responses_df_2017[question_of_interest_cell18].replace(
    {
        'United States': 'United States of America',
        "People 's Republic of China": 'China',
        'United Kingdom': 'United Kingdom of Great Britain and Northern Ireland'
    },
    inplace=True
)

# 4) Bucket any non-subset country value into 'Other'
#    (use the same chained‐indexing pattern as original code to match side‐effects)
for df in [
    responses_df_2017,
    responses_df_2018_cell10,
    responses_df_2019_cell10,
    responses_df_2020,
    responses_df_2021,
    responses_df_2022_cell10
]:
    df[question_of_interest_cell18][
        ~df[question_of_interest_cell18].isin(subset_of_countries)
    ] = 'Other'

# 5) Combine data from multiple years into one DataFrame
def combine_subset_of_data_from_multiple_years_18(
    question_of_interest,
    x_axis_title,
    include_2017=False
):
    year_to_df = {
        '2018': responses_df_2018_cell10,
        '2019': responses_df_2019_cell10,
        '2020': responses_df_2020,
        '2021': responses_df_2021,
        '2022': responses_df_2022_cell10
    }
    years = ['2018', '2019', '2020', '2021', '2022']
    if include_2017:
        year_to_df['2017'] = responses_df_2017
        years.insert(0, '2017')

    dfs = []
    axis_col = x_axis_title if x_axis_title else question_of_interest
    for year in years:
        pct = (
            count_then_return_percent_18(
                year_to_df[year], question_of_interest
            )
            .sort_index()
        )
        temp = pct.reset_index()
        temp.columns = [axis_col, 'percentage']
        temp['year'] = year
        dfs.append(temp)

    df_combined = pd.concat(dfs, ignore_index=True)
    return df_combined[['percentage', 'year', axis_col]]

# 6) Execute the combine and final cleanups
a_country_df = combine_subset_of_data_from_multiple_years_18(
    question_of_interest_cell18,
    title_for_x_axis,
    include_2017=True
)
country_df_combined_cell18 = (
    a_country_df
    .sort_values(by=['year', 'percentage'], ascending=True)
)
# Collapse the long UK name back to the shorter form
country_df_combined_cell18.replace(
    'United Kingdom of Great Britain and Northern Ireland',
    'United Kingdom',
    inplace=True
)

# 7) Display info to confirm success
country_df_combined_cell18.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 309 entries, 2 to 271
Data columns (total 3 columns):
 #   Column                                     Non-Null Count  Dtype
---  ------                                     --------------  -----
 0   percentage                                 309 non-null    float64
 1   year                                       309 non-null    object
 2   In which country do you currently reside?  309 non-null    object
dtypes: float64(1), object(2)
memory usage: 11.1+ KB
CPU times: user 1.12 s, sys: 15.4 ms, total: 1.14 s
Wall time: 1.13 s


In [5]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high/checkpoints/post_cell_6_try_4.pickle

migration speed (bps): 701649514.7648044
---------------------------
variables to migrate:
orig_output 16
responses_df_2022 48
file_name_2019 78
file_name_2018 76
responses_df_2021 48
directory 163
responses_df_2017 48
count_then_return_percent_18 144
base_dir_2017 155
base_dir_2019 155
base_dir_2018 155
responses_df_2019 48
responses_df_2018 48
directory_cell8 170
responses_df_2019_cell10 48
responses_df_2018_cell10 48
replace_hyphen_with_en_dash 144
responses_df_2022_cell10 48
base_dir_2021 155
glob 72
percentages_per_country_df 48
create_dataframe_of_counts_16 144
df 48
pd 72
title_for_x_axis 49
sns 72
percentages_cell17 48
question_of_interest_cell18 90
np 72
question_of_interest 90
base_dir_2022 155
warnings 72
count_then_return_percent_17 144
combine_subset_of_data_from_multiple_years_18 144
go 72
orientation_for_chart 50
px 72
responses_in_order 184
percentages 48
file_name_2022 81
responses_df_2020 48
file_name_2021 81
load_survey_data 144
base_dir_2020 155
a_country_df 48
file

In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables []
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['responses_df_2020', 'responses_df_2018', 'responses_df_2021', 'responses_df_2019', 'responses_df_2022']
Intermediate variables ['file_name_2020', 'file_name_2019', 'file_name_2022', 'file_name_2018', 'base_dir_2021', 'file_name_2017', 'responses_df_2017', 'base_dir_2018', 'base_dir_2017', 'directory', 'base_dir_2022', 'file_name_2021', 'base_dir_2019', 'base_dir_2020', 'directory_cell8']
Future variables ['responses_df_2022_cell10', 'responses_df_2017']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables ['responses_df_2022', 'responses_df_2019_cell10', 'responses_df_2022_cell10']
Intermediate variables ['responses_df_2018_cell10']
Future variables ['responses_df_2021', 'responses_df_2018_cell10', 'responses_df_2022_cell10', 'responses_df_2017']
Modified dataframes
  respon

In [7]:

with open("/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_6_try_4.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[6], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
responses_df_2020_opt = responses_df_2020
responses_df_2021_opt = responses_df_2021
subset_of_countries_opt = subset_of_countries
title_for_x_axis_opt = title_for_x_axis
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated/checkpoints/post_cell_6.pickle
assert compare_df(responses_df_2020_opt, responses_df_2020)
assert compare_df(responses_df_2021_opt, responses_df_2021)
assert subset_of_countries_opt == subset_of_countries
assert title_for_x_axis_opt == title_for_x_axis

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
